# Day 02: Deep Autoencoder Bằng PyTorch
### Sổ Tay Thực Hành: `autoencoder_pytorch.ipynb`

> **Bài báo gốc:** *Reducing the Dimensionality of Data with Neural Networks*  
> **Tác giả:** G. E. Hinton & R. R. Salakhutdinov (Science, 2006)  
> **Tài liệu tham khảo:** [`resources/Day 2 Autoencoder.pdf`](./resources/Day%202%20Autoencoder.pdf)

---

## 🎯 Mục Tiêu Cốt Lõi Của Notebook
1. **Giải thích chi tiết cho người mới bắt đầu:** Hiểu rõ kiến trúc Bộ mã hóa (Encoder) $\to$ Nút thắt cổ chai (Bottleneck) $\to$ Bộ giải mã (Decoder), và lý do các kỹ thuật Deep Learning hiện đại (ReLU, BatchNorm, Adam) cho phép huấn luyện Deep Autoencoder trực tiếp mà không cần qua RBM Pre-training.
2. **Cài đặt 3 kiến trúc Autoencoder trong PyTorch:**
   - **Shallow Autoencoder (1986):** Mạng nơ-ron nông 1 tầng ẩn.
   - **Deep Autoencoder (Hinton 2006):** Mạng sâu $784 \to 1000 \to 500 \to 250 \to 2$.
   - **Convolutional Autoencoder (CAE):** Kiến trúc tích chập 2D hiện đại.
3. **Thực nghiệm đối chuẩn giá trị học thuật:** Tái hiện và so sánh định lượng chất lượng nén ảnh giữa 4 thế hệ: Linear PCA (1901) $\to$ Shallow AE (1986) $\to$ Deep AE (2006) $\to$ Conv AE.

## 💡 Lý Thuyết Cho Người Mới Bắt Đầu: Nguyên Lý Autoencoder & Nút Thắt Cổ Chai

- **Encoder $f_\theta$:** Ánh xạ dữ liệu nhiều chiều $x \in \mathbb{R}^{784}$ về vector tiềm ẩn $z \in \mathbb{R}^d$ ($d \ll 784$):
  $$z = f_\theta(x)$$
- **Bottleneck (Code Layer):** Vùng thắt lại của mạng, giới hạn dung lượng thông tin để ép mô hình chắt lọc các đặc trưng cốt lõi nhất của đa tạp dữ liệu thay vì học vẹt.
- **Decoder $g_\phi$:** Tái tạo lại dữ liệu gốc từ $z$:
  $$\hat{x} = g_\phi(z)$$
- **Hàm mất mát tái tạo (Reconstruction Loss):**
  $$\mathcal{L}(x, \hat{x}) = \frac{1}{2} \|x - \hat{x}\|^2$$

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Đang thực thi trên thiết bị: {device}")

## PHẦN 1: ĐỊNH NGHĨA CÁC KIẾN TRÚC MÔ HÌNH NƠ-RON (PYTORCH)

In [ ]:
class ShallowAutoencoder(nn.Module):
    """Mô hình Autoencoder nông 1 tầng ẩn (Rumelhart 1986)"""
    def __init__(self, input_dim=784, latent_dim=2):
        super(ShallowAutoencoder, self).__init__()
        self.encoder = nn.Linear(input_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


class DeepAutoencoder(nn.Module):
    """Mô hình Deep Autoencoder theo sát bài báo Hinton & Salakhutdinov (Science 2006)"""
    def __init__(self, input_dim=784, latent_dim=2):
        super(DeepAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1000),
            nn.BatchNorm1d(1000),
            nn.ReLU(inplace=True),
            nn.Linear(1000, 500),
            nn.BatchNorm1d(500),
            nn.ReLU(inplace=True),
            nn.Linear(500, 250),
            nn.BatchNorm1d(250),
            nn.ReLU(inplace=True),
            nn.Linear(250, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 250),
            nn.BatchNorm1d(250),
            nn.ReLU(inplace=True),
            nn.Linear(250, 500),
            nn.BatchNorm1d(500),
            nn.ReLU(inplace=True),
            nn.Linear(500, 1000),
            nn.BatchNorm1d(1000),
            nn.ReLU(inplace=True),
            nn.Linear(1000, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


class ConvAutoencoder(nn.Module):
    """Mô hình Convolutional Autoencoder (CAE) hiện đại cho dữ liệu ảnh 2D"""
    def __init__(self, latent_dim=2):
        super(ConvAutoencoder, self).__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.fc_encode = nn.Linear(32 * 7 * 7, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 32 * 7 * 7)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x_2d = x.view(-1, 1, 28, 28)
        feat = self.encoder_conv(x_2d)
        z = self.fc_encode(feat.view(feat.size(0), -1))
        feat_dec = nn.functional.relu(self.fc_decode(z)).view(-1, 32, 7, 7)
        recon_2d = self.decoder_conv(feat_dec)
        return recon_2d.view(-1, 784), z

print("[✓] Đã định nghĩa thành công ShallowAutoencoder, DeepAutoencoder và ConvAutoencoder!")

## PHẦN 2: NẠP TẬP DỮ LIỆU & BỘ SINH DỮ LIỆU FALLBACK AN TOÀN

In [ ]:
num_samples = 1600
try:
    from torchvision import datasets, transforms
    transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: torch.flatten(x))])
    mnist = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    loader = DataLoader(mnist, batch_size=num_samples, shuffle=True)
    all_images, all_labels = next(iter(loader))
    print(f"[*] Đã nạp thành công {num_samples} ảnh chuẩn MNIST (784 chiều)!")
except Exception as e:
    print(f"[!] Chuyển sang dữ liệu đa tạp nhân tạo Fallback ({e})...")
    np.random.seed(42)
    t1 = np.random.uniform(-np.pi, np.pi, size=(num_samples, 1))
    t2 = np.random.uniform(-np.pi, np.pi, size=(num_samples, 1))
    all_labels = torch.tensor(np.random.randint(0, 10, size=num_samples), dtype=torch.long)
    X_synth = np.zeros((num_samples, 784), dtype=np.float32)
    for i in range(784):
        X_synth[:, i] = np.sin((i % 28 + 1) * 0.2 * t1[:, 0]) * np.cos((i // 28 + 1) * 0.1 * t2[:, 0])
    X_synth = (X_synth - X_synth.min()) / (X_synth.max() - X_synth.min() + 1e-8)
    all_images = torch.tensor(X_synth, dtype=torch.float32)

split = 1200
train_x, test_x = all_images[:split], all_images[split:]
train_y, test_y = all_labels[:split], all_labels[split:]
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True)
print(f"[*] Kích thước dữ liệu: Train = {train_x.shape}, Test = {test_x.shape}")

## PHẦN 3: HUẤN LUYỆN & ĐỐI CHUẨN 4 CỘT MỐC CÔNG NGHỆ

Chúng ta sẽ đối chuẩn định lượng khả năng nén từ 784 chiều về **2 chiều** giữa:
1. **Linear PCA (1901)**
2. **Shallow Autoencoder (1986)**
3. **Deep Autoencoder (Hinton 2006)**
4. **Convolutional Autoencoder (Hiện Đại)**

In [ ]:
models_mse = {}
loss_histories = {}
latent_dim = 2

# 1. Linear PCA (Pearson 1901)
t0 = time.time()
pca = PCA(n_components=latent_dim)
pca.fit(train_x.numpy())
pca_recon = pca.inverse_transform(pca.transform(test_x.numpy()))
pca_mse = float(np.mean((test_x.numpy() - pca_recon) ** 2))
pca_time = time.time() - t0
models_mse["1. Linear PCA (1901)"] = pca_mse

# Hàm tiện ích huấn luyện PyTorch
def train_model(model, epochs=15, lr=0.003):
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    crit = nn.MSELoss()
    hist = []
    for ep in range(1, epochs + 1):
        model.train()
        tot = 0.0
        for bx, _ in train_loader:
            bx = bx.to(device)
            opt.zero_grad()
            rx, _ = model(bx)
            loss = crit(rx, bx)
            loss.backward()
            opt.step()
            tot += loss.item()
        hist.append(tot / len(train_loader))
    return hist

# 2. Shallow Autoencoder (1986)
t0 = time.time()
shallow = ShallowAutoencoder(784, latent_dim)
loss_histories["Shallow AE"] = train_model(shallow, epochs=15, lr=0.005)
shallow.eval()
with torch.no_grad():
    sh_recon, _ = shallow(test_x.to(device))
    sh_mse = nn.MSELoss()(sh_recon, test_x.to(device)).item()
models_mse["2. Shallow AE (1986)"] = sh_mse
sh_time = time.time() - t0

# 3. Deep Autoencoder (Hinton 2006)
t0 = time.time()
deep = DeepAutoencoder(784, latent_dim)
loss_histories["Deep AE (Hinton)"] = train_model(deep, epochs=20, lr=0.002)
deep.eval()
with torch.no_grad():
    deep_recon, deep_latents = deep(test_x.to(device))
    dp_mse = nn.MSELoss()(deep_recon, test_x.to(device)).item()
models_mse["3. Deep AE (Hinton)"] = dp_mse
dp_time = time.time() - t0

# 4. Conv Autoencoder (Hiện Đại)
t0 = time.time()
conv = ConvAutoencoder(latent_dim)
loss_histories["Conv AE (Modern)"] = train_model(conv, epochs=20, lr=0.003)
conv.eval()
with torch.no_grad():
    conv_recon, _ = conv(test_x.to(device))
    cv_mse = nn.MSELoss()(conv_recon, test_x.to(device)).item()
models_mse["4. Conv AE (Modern)"] = cv_mse
cv_time = time.time() - t0

# BẢNG ĐỐI CHUẨN ĐỊNH LƯỢNG
print("=" * 90)
print(f"{'Mô Hình / Giải Thuật':<28} | {'Bottleneck':<10} | {'Test MSE':<12} | {'Cải thiện vs PCA':<18} | Thời Gian")
print("=" * 90)
print(f"{'1. Linear PCA (1901)':<28} | {'2 chiều':<10} | {pca_mse:10.5f}   | {'Baseline (0.0%)':<18} | {pca_time:7.2f}s")
print(f"{'2. Shallow AE (1986)':<28} | {'2 chiều':<10} | {sh_mse:10.5f}   | {((pca_mse - sh_mse)/pca_mse * 100):+15.2f}% | {sh_time:7.2f}s")
print(f"{'3. Deep AE (Hinton 2006)':<28} | {'2 chiều':<10} | {dp_mse:10.5f}   | {((pca_mse - dp_mse)/pca_mse * 100):+15.2f}% | {dp_time:7.2f}s")
print(f"{'4. Conv AE (Hiện Đại)':<28} | {'2 chiều':<10} | {cv_mse:10.5f}   | {((pca_mse - cv_mse)/pca_mse * 100):+15.2f}% | {cv_time:7.2f}s")
print("=" * 90)

## PHẦN 4: TRỰC QUAN HÓA TOÀN DIỆN 4 Ô ĐỒ THỊ ĐỐI CHUẨN HỌC THUẬT

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.25)

# 1. So sánh ảnh tái tạo
sub_gs = gs[0, 0].subgridspec(5, 8, hspace=0.05, wspace=0.05)
rows = [
    ("Ảnh Gốc", test_x[:8].numpy()),
    ("PCA", pca_recon[:8]),
    ("Shallow AE", sh_recon[:8].cpu().numpy()),
    ("Deep AE", deep_recon[:8].cpu().numpy()),
    ("Conv AE", conv_recon[:8].cpu().numpy())
]
for r_i, (nm, dt) in enumerate(rows):
    for c_i in range(8):
        ax = fig.add_subplot(sub_gs[r_i, c_i])
        ax.imshow(dt[c_i].reshape(28, 28), cmap="gray")
        ax.axis("off")
        if c_i == 0:
            ax.text(-5, 14, nm, va='center', ha='right', fontsize=9, fontweight='bold')

ax_t = fig.add_subplot(gs[0, 0])
ax_t.axis("off")
ax_t.set_title("1. So Sánh Chất Lượng Tái Tạo Ảnh (Reconstruction)", fontsize=11, fontweight='bold', pad=15)

# 2. Biểu đồ thanh Test MSE
ax2 = fig.add_subplot(gs[0, 1])
bars = ax2.bar(list(models_mse.keys()), list(models_mse.values()), color=["#7f7f7f", "#d95f02", "#1f77b4", "#2ca02c"], width=0.45, edgecolor="black", alpha=0.85)
ax2.set_ylabel("Sai Số Bình Phương Trung Bình (Test MSE)", fontsize=10, fontweight="bold")
ax2.set_title("2. Sai Số Tái Tạo Nén Về 2 Chiều", fontsize=11, fontweight="bold")
ax2.grid(True, linestyle="--", alpha=0.5)
for b in bars:
    h = b.get_height()
    ax2.text(b.get_x() + b.get_width() / 2., h + 0.001, f"{h:.4f}", ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. Không gian tiềm ẩn 2D
ax3 = fig.add_subplot(gs[1, 0])
latents_np = deep_latents.cpu().numpy()
sc = ax3.scatter(latents_np[:, 0], latents_np[:, 1], c=test_y.numpy(), cmap="tab10", alpha=0.7, s=14)
cbar = plt.colorbar(sc, ax=ax3, ticks=range(10))
cbar.set_label("Nhãn Chữ Số (0-9)", fontsize=9)
ax3.set_xlabel("Thành phần tiềm ẩn $z_1$", fontsize=10, fontweight="bold")
ax3.set_ylabel("Thành phần tiềm ẩn $z_2$", fontsize=10, fontweight="bold")
ax3.set_title("3. Không Gian Tiềm Ẩn 2D Deep AE (Figure 3 - Hinton 2006)", fontsize=11, fontweight="bold")
ax3.grid(True, linestyle="--", alpha=0.4)

# 4. Đường cong mất mát
ax4 = fig.add_subplot(gs[1, 1])
for nm, hst in loss_histories.items():
    ax4.plot(range(1, len(hst) + 1), hst, marker='o', markersize=4, linewidth=2, label=nm)
ax4.set_xlabel("Epochs", fontsize=10, fontweight="bold")
ax4.set_ylabel("Mất Mát Tái Tạo (MSE Loss)", fontsize=10, fontweight="bold")
ax4.set_title("4. Đường Cong Hội Tụ Huấn Luyện (Loss vs Epochs)", fontsize=11, fontweight="bold")
ax4.grid(True, linestyle="--", alpha=0.5)
ax4.legend(fontsize=9)

plt.show()

## 🎓 ĐÚC KẾT HỌC THUẬT DÀNH CHO NGƯỜI MỚI BẮT ĐẦU

1. **Bảo toàn hình học phi tuyến:**
   - PCA chỉ có thể chiếu tuyến tính, do đó khi nén về 2 chiều sẽ đè bẹp các cụm chữ số khác nhau vào cùng một vùng trung tâm.
   - Deep Autoencoder của Hinton uốn lượn theo đa tạp dữ liệu phi tuyến, phân tách 10 lớp chữ số thành các nhánh rõ rệt trong không gian 2D.
2. **Sức mạnh của độ sâu (Depth):**
   - Shallow Autoencoder (1 tầng ẩn) chỉ tương đương với xấp xỉ không gian con của PCA, không thể tái tạo sắc nét.
   - Deep Autoencoder với kiến trúc phân tầng sâu trích xuất được các biểu diễn đặc trưng trừu tượng theo cấp bậc (cạnh $\to$ góc $\to$ vòng $\to$ toàn thể chữ số).
3. **Bước tiến của Convolutional Autoencoder (CAE):**
   - CAE khai thác cấu trúc 2D không gian và chia sẻ trọng số, giảm mạnh số tham số so với MLP và tái tạo đường nét chữ số mượt mà hơn.